In [ ]:
import pandas as pd
import numpy as np
from enum import Enum
import cv2
import json

def obtain_estimates(video_ID, start_entry_ratio, end_exit_ratio):
    vid = cv2.VideoCapture(f'/home/edisonz/maneuver_heuristics25/Parking-clip{video_ID}.mp4')
    frame_width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))  # Max x-coordinate
    frame_height = 2*int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))//3  # Max y-coordinate
    # ----------------------------------------------------------------------------------------------------------------------------

    # Enter hyperparameters (adjustable)
    start_entry_ratio = 0.5 # 0 = zone based | 1 = peak based 
    end_entry_ratio = 1.0 # 0 = front parking zone | 1 = rear parking zone

    # Exit hyperparameters (adjustable)
    start_exit_ratio = 0 # 0 = rear parking zone | 1 = front parking zone
    end_exit_ratio = 1.0 # 0 = peak based | 1 = zone based

    # low motion hyperparameters 
    movement_threshold = 1.0

    # Peak hyperparameters 
    tA = 5
    aA = np.pi/6

    def load_lines(json_file):
        with open(json_file, 'r') as f:
            data = json.load(f)
        lines = [list(map(tuple, line)) for line in data['lines']]
        loaded_lines = []
        for line in lines: 
            m = (line[1][1] - line[0][1])/(line[1][0] - line[0][0])
            c = line[0][1] - m * line[0][0]
            loaded_lines.append((m, c))
        return loaded_lines

    # Load data
    vehicle_df = pd.read_csv(f'clips/Parking-clip{video_ID}.csv')
    maneuvers = pd.read_csv(f'clip-annotations/maneuver{video_ID}.csv')
    start_frame, end_frame = np.median(maneuvers["start_frame"]), np.median(maneuvers["end_frame"])
    driving_df = vehicle_df[vehicle_df["in_parking_zone"]==0]
    driving_coords = driving_df[['x', 'y']].values
    parking_df = vehicle_df[vehicle_df["in_parking_zone"]==1]
    parking_coords = parking_df[['x', 'y']].values
    start_frame = int(round(start_frame))
    end_frame = int(round(end_frame))

    region_code = None
    with open("mappings.json", 'r') as f:
        data = json.load(f)
        region_code = data.get(f'{video_ID}')
    parking_lines = f'lines{region_code}.json'

    loaded_lines = load_lines(parking_lines)

    class ManeuverType(Enum):
        ENT = 0
        EXT = 1

    maneuver_type = ManeuverType.ENT if video_ID[-3:].lower()=='ent' else ManeuverType.EXT

    # --------------------------------------------------------------------------------------------------------------------------

    # Determine if direction reversal in x or y occured
    def is_reversal(r1, r2):
        angle_diff = np.abs(np.arctan2(r1[1], r1[0]) - np.arctan2(r2[1], r2[0]))
        if angle_diff < aA:
            return False  # No significant angle change
        return (r1[0] * r2[0] < 0) or (r1[1] * r2[1] < 0)  # Opposite signs in x or y

    # Determine if vehicle is hugging the border of the frame
    def on_border(df_row):
        height = df_row["height"]
        width = df_row["width"]
        x = df_row["x"]
        y = df_row["y"]
        left_x = int(x-width//2)
        right_x = int(x+width//2)
        top_y = int(y-height//2)
        bottom_y = int(y+height//2)
        return (left_x==0 or top_y==0 or right_x==frame_width or bottom_y==frame_height)

    def find_shortest_distance(point, loaded_lines):
        """
        Compute the shortest distance for a given point:
        - If the point lies between two parking lines, return the sum of perpendicular distances to the nearest lines on either side.
        - If the point does not lie between two lines, return the perpendicular distance to the closest line (min perp dist among all lines).
        Lines include both original from json and extrapolated ones.
        
        Args:
            point (tuple): (x, y) coordinates of the point
            json_file (str): Path to the JSON file containing line data
        
        Returns:
            float: Shortest distance (sum of perpendicular distances or min distance to closest line), or None if no lines exist
        """
        x_point, y_point = point
    
        # Original lines
        intersections = []
        for m, c in loaded_lines:
            x_inter = (y_point - c) / m
            intersections.append(x_inter)
    
        # Separate intersections into left and right
        left = [(x, params) for x, params in zip(intersections, loaded_lines) if x < x_point]
        right = [(x, params) for x, params in zip(intersections, loaded_lines) if x > x_point]
    
        # Compute perpendicular distance to a line
        def point_to_line_distance(point, line_params):
            x_p, y_p = point
            if line_params[1] is None:  # Vertical line: x = c
                return abs(x_p - line_params[0])
            m, c = line_params
            return abs(m * x_p - y_p + c) / np.sqrt(m**2 + 1)
    
        # Case 1: Point lies between two lines (considering all)
        if left and right:
            __, left_params = max(left, key=lambda x: x[0])
            __, right_params = min(right, key=lambda x: x[0])
            d_left = point_to_line_distance(point, left_params)
            d_right = point_to_line_distance(point, right_params)
            return d_left + d_right
    
        # Case 2: Point does not lie between two lines, find closest line by min perp dist
        if left and not right:
            __, left_params = max(left, key=lambda x: x[0])
            return point_to_line_distance(point, left_params)
        if right and not left:
            __, right_params = min(right, key=lambda x: x[0])
            return point_to_line_distance(point, right_params) 

    def in_adjacent_zones(p1, p2, loaded_lines):
        """This function calculates whether two points are in the same or adjacent zones"""
        x_point1 = p1[0]
        y_point1 = p1[1]
        x_point2 = p2[0]
        y_point2 = p2[1]

        intersections1 = []
        intersections2 = []
        for m, c in loaded_lines:
            x_inter1 = (y_point1 - c) / m
            intersections1.append(x_inter1)
            x_inter2 = (y_point2 - c) / m
            intersections2.append(x_inter2)

        left1 = [(x, params) for x, params in zip(intersections1, loaded_lines) if x < x_point1]
        right1 = [(x, params) for x, params in zip(intersections1, loaded_lines) if x > x_point1]

        left2 = [(x, params) for x, params in zip(intersections2, loaded_lines) if x < x_point2]
        right2 = [(x, params) for x, params in zip(intersections2, loaded_lines) if x > x_point2]

        if len(left1) > 0: 
            __, left_params1 = max(left1, key = lambda x: x[0])  
        else: 
            left_params1 = None

        if len(right1) > 0:
            __, right_params1 = min(right1, key = lambda x: x[0]) 
        else:
            right_params1 = None

        if len(left2) > 0: 
            __, left_params2 = max(left2, key = lambda x: x[0])  
        else: 
            left_params2 = None

        if len(right2) > 0:
            __, right_params2 = min(right2, key = lambda x: x[0]) 
        else:
            right_params2 = None

        return (left_params1 == left_params2 or left_params1 == right_params2 or 
                right_params1 == right_params2 or right_params1 == left_params2)

    if maneuver_type == ManeuverType.ENT:
        # --- End-of-Maneuver Detection (STOP) ---
        front_end_point = parking_coords[0]
        dists_from_front = np.linalg.norm(parking_coords - front_end_point, axis=1)

        rear_end_idx = np.argmax(dists_from_front)
        rear_end_point = parking_coords[rear_end_idx]
        dists_from_rear = np.linalg.norm(parking_coords - rear_end_point, axis=1)
        close_indices = np.where(dists_from_rear <= movement_threshold)[0]
        candidate_frames = parking_df.iloc[close_indices]['frame']
        adjusted_rear_idx = close_indices[np.argmin(candidate_frames)]
        rear_end_point = parking_coords[adjusted_rear_idx]

        interpolated_point = front_end_point*(1-end_entry_ratio) + rear_end_point*end_entry_ratio

        dists_interpolated = np.linalg.norm(parking_coords - interpolated_point, axis=1)
        final_end_idx = np.argmin(dists_interpolated)
        final_end_frame = parking_df.iloc[final_end_idx]['frame']

        # --- Start-of-Maneuver Detection ---
        peak_idx = None
        consecutive_post_peak_frames=0
        for idx in range(tA, len(driving_coords)-tA):
            r1 = driving_coords[idx] - driving_coords[idx-tA]
            r2 = driving_coords[idx+tA] - driving_coords[idx]
            if is_reversal(r1, r2):
                true_reversal = True
                for t in range(1, tA):
                    rA = driving_coords[idx] - driving_coords[max(0, idx-tA-t)]
                    rB = driving_coords[min(len(driving_coords)-1, idx+tA+t)] - driving_coords[idx]
                    if not is_reversal(rA, rB):
                        true_reversal = False
                        break
                if true_reversal and in_adjacent_zones(front_end_point, driving_coords[idx], loaded_lines):      
                    peak_idx = idx    
            if peak_idx is not None and not is_reversal(r1, r2):
                consecutive_post_peak_frames+=1
            else:
                consecutive_post_peak_frames=0
            if consecutive_post_peak_frames>=tA:
                break

        if(peak_idx is None):
            consecutive_turn_frames = 0
            for idx in range(len(driving_coords)-1, 0, -1):
                aspect_ratio1 = driving_df.iloc[idx-1]['height']/driving_df.iloc[idx-1]['width']
                aspect_ratio2 = driving_df.iloc[idx]['height']/driving_df.iloc[idx]['width']
                if(round(aspect_ratio1, 2)!=round(aspect_ratio2, 2)):
                    consecutive_turn_frames+=1
                if(consecutive_turn_frames==tA):
                    peak_idx = idx

        if on_border(driving_df.iloc[peak_idx]):
            for idx in range(peak_idx, len(driving_df)):
                if not on_border(driving_df.iloc[idx]):
                    peak_idx = idx
                    break
                
        peak_point = driving_coords[peak_idx]

        bottom_peak_point = peak_point.copy()
        bottom_peak_point[1]+=driving_df.iloc[peak_idx]['height']//2

        threshold_distance = find_shortest_distance(bottom_peak_point, loaded_lines) * (1-start_entry_ratio)
        final_start_idx = 0
        accumulated_distance = 0
        for i in range(peak_idx - 1, -1, -1):
            if threshold_distance is None:
                break

            accumulated_distance+=np.linalg.norm(driving_coords[i+1]-driving_coords[i])
            if(accumulated_distance>=threshold_distance and
               np.linalg.norm(driving_coords[i]-driving_coords[peak_idx])>=threshold_distance):
                final_start_idx = i
                break     

        final_start_frame = driving_df.iloc[final_start_idx]["frame"]
        return final_end_frame - final_start_frame

    if(maneuver_type==ManeuverType.EXT):
        # detect start of exit maneuver 
        front_start_point = driving_coords[0]
        dists_from_front = np.linalg.norm(parking_coords - front_start_point, axis=1)

        rear_start_idx = np.argmax(dists_from_front)
        rear_start_point = parking_coords[rear_start_idx]

        dists_from_rear = np.linalg.norm(parking_coords - rear_start_point, axis=1)
        close_indices = np.where(dists_from_rear <= movement_threshold)[0]
        candidate_frames = parking_df.iloc[close_indices]['frame']
        adjusted_rear_idx = close_indices[np.argmax(candidate_frames)]
        rear_start_point = parking_coords[adjusted_rear_idx]

        interpolated = front_start_point*(start_exit_ratio) + rear_start_point*(1-start_exit_ratio)

        dists_interpolated = np.linalg.norm(parking_coords - interpolated, axis=1)
        final_start_idx = np.argmin(dists_interpolated)
        final_start_frame = parking_df.iloc[final_start_idx]['frame']

        # detect end of exit maneuver
        peak_idx = None
        consecutive_post_peak_frames=0
        for idx in range(tA, len(driving_coords)-tA):
            r1 = driving_coords[idx] - driving_coords[idx-tA]
            r2 = driving_coords[idx+tA] - driving_coords[idx]
            if is_reversal(r1, r2):
                true_reversal = True
                for t in range(1, tA):
                    rA = driving_coords[idx] - driving_coords[max(0, idx-tA-t)]
                    rB = driving_coords[min(len(driving_coords)-1, idx+tA+t)] - driving_coords[idx]
                    if not is_reversal(rA, rB):
                        true_reversal = False
                        break
                if true_reversal and in_adjacent_zones(front_start_point, driving_coords[idx], loaded_lines):      
                    peak_idx = idx    
            if peak_idx is not None and not is_reversal(r1, r2):
                consecutive_post_peak_frames+=1
            else:
                consecutive_post_peak_frames=0
            if consecutive_post_peak_frames>=tA:
                break

        if(peak_idx is None):
            peak_idx = 0
            consecutive_turn_frames = 0
            for idx in range(len(driving_coords)-1):
                aspect_ratio1 = driving_df.iloc[idx]['height']/driving_df.iloc[idx]['width']
                aspect_ratio2 = driving_df.iloc[idx+1]['height']/driving_df.iloc[idx+1]['width']
                if(round(aspect_ratio1, 2)!=round(aspect_ratio2, 2)):
                    consecutive_turn_frames+=1
                if(consecutive_turn_frames==tA):
                    peak_idx = idx

        if on_border(driving_df.iloc[peak_idx]):
            for idx in range(peak_idx-1, -1, -1):
                if not on_border(driving_df.iloc[idx]):
                    peak_idx = idx
                    break
                
        peak_point = driving_coords[peak_idx]

        bottom_peak_point = peak_point.copy()
        bottom_peak_point[1]+=driving_df.iloc[peak_idx]['height']//2

        threshold_distance = find_shortest_distance(bottom_peak_point, loaded_lines) * end_exit_ratio

        final_end_idx = -1
        accumulated_distance = 0
        for i in range(peak_idx, len(driving_coords)-1):
            accumulated_distance+=np.linalg.norm(driving_coords[i+1]-driving_coords[i])
            if(accumulated_distance>=threshold_distance and
               np.linalg.norm(driving_coords[i+1]-driving_coords[peak_idx])>=threshold_distance):
                final_end_idx = i+1
                break

        final_end_frame = driving_df.iloc[final_end_idx]["frame"]

        return final_end_frame - final_start_frame


In [31]:
enter_IDs = ['1ENT', '3ENT', '9ENT', '11ENT', '21ENT', '90ENT', '91ENT', '93ENT',
             '94ENT', '95ENT', '96ENT', '97ENT', '98ENT', '290ENT', '291ENT', '292ENT']
exit_IDs = ['0EXT', '2EXT', '4EXT', '80EXT', '81EXT', '83EXT', '84EXT', '85EXT', '86EXT']

In [ ]:
enter_annotations = []
enter_predictions = []
absolute_errors_ent = []
for id in enter_IDs:
  #print(id)
  predicted_time = obtain_estimates(id, 0, 1)
  maneuvers = pd.read_csv(f'clip-annotations/maneuver{id}.csv')
  enter_time = np.median(maneuvers["end_frame"]) - np.median(maneuvers["start_frame"])
  enter_annotations.append(enter_time)
  enter_predictions.append(predicted_time)
  #print(f'annotated time: {enter_time/30.0}')
  #print(f'predicted time: {predicted_time/30.0}')
  absolute_errors_ent.append(np.abs(enter_time-predicted_time)/30.0)

enter_predictions = np.array(enter_predictions)
absolute_errors_ent = np.array(absolute_errors_ent)

print(f'interval = {np.mean(enter_predictions)/30.0}+/-{np.mean(absolute_errors_ent)}')

1ENT
annotated time: 5.566666666666666
predicted time: 5.6
3ENT
annotated time: 3.566666666666667
predicted time: 6.6
9ENT
annotated time: 5.966666666666667
predicted time: 6.4
11ENT
annotated time: 9.3
predicted time: 9.9
21ENT
annotated time: 8.6
predicted time: 10.0
90ENT
annotated time: 4.3
predicted time: 3.6
91ENT
annotated time: 5.133333333333334
predicted time: 4.5
93ENT
annotated time: 3.466666666666667
predicted time: 3.4
94ENT
annotated time: 4.466666666666667
predicted time: 3.6
95ENT
annotated time: 6.5
predicted time: 6.3
96ENT
annotated time: 7.866666666666666
predicted time: 8.4
97ENT
annotated time: 3.566666666666667
predicted time: 2.7
98ENT
annotated time: 5.766666666666667
predicted time: 4.8
290ENT
annotated time: 7.933333333333334
predicted time: 7.6
291ENT
annotated time: 9.1
predicted time: 9.6
292ENT
annotated time: 5.833333333333333
predicted time: 5.0
interval = 6.125+/-0.75


In [ ]:
exit_annotations = []
exit_predictions = []
absolute_errors_ext = []
for id in exit_IDs:
  #print(id)
  predicted_time = obtain_estimates(id, 0, 1)
  maneuvers = pd.read_csv(f'clip-annotations/maneuver{id}.csv')
  exit_time = np.median(maneuvers["end_frame"]) - np.median(maneuvers["start_frame"])
  exit_annotations.append(exit_time)
  exit_predictions.append(predicted_time)
  #print(f'annotated time: {exit_time/30.0}')
  #print(f'predicted time: {predicted_time/30.0}')
  absolute_errors_ext.append(np.abs(exit_time-predicted_time)/30.0)

exit_predictions = np.array(exit_predictions)
absolute_errors_ext = np.array(absolute_errors_ext)

print(f'interval = {np.mean(exit_predictions)/30.0}+/-{np.mean(absolute_errors_ext)}')

0EXT
1.0
120.0
576.0
annotated time: 15.266666666666667
predicted time: 15.2
2EXT
1.0
201.0
657.0
annotated time: 10.466666666666667
predicted time: 15.2
4EXT
1.0
219.0
579.0
annotated time: 9.066666666666666
predicted time: 12.0
80EXT
1.0
3.0
300.0
annotated time: 9.433333333333334
predicted time: 9.9
81EXT
1.0
27.0
312.0
annotated time: 9.433333333333334
predicted time: 9.5
83EXT
1.0
165.0
537.0
annotated time: 15.5
predicted time: 12.4
84EXT
1.0
99.0
399.0
annotated time: 9.833333333333334
predicted time: 10.0
85EXT
1.0
57.0
618.0
annotated time: 16.366666666666667
predicted time: 18.7
86EXT
1.0
45.0
543.0
annotated time: 15.4
predicted time: 16.6
interval = 13.277777777777777+/-1.674074074074074
